In [11]:
import os
import subprocess

# ── Write the LaTeX ───────────────────────────────────────────
latex_content = r'''
\documentclass[12pt,a4paper]{article}

\usepackage[margin=1in]{geometry}
\usepackage{amsmath,amssymb}
\usepackage{graphicx}
\usepackage{hyperref}
\usepackage{booktabs}
\usepackage{caption}
\usepackage{subcaption}
\usepackage[numbers]{natbib}
\usepackage{float}
\usepackage{xcolor}
\usepackage{graphicx}

% Tell LaTeX to look for images in the "figures" folder
\graphicspath{{figures/}}

\title{\textbf{EcoEcon: Ecological Resilience Indicators for Systemic Risk---A Retrospective Analysis of the 2008 Financial Crisis with a 2020 Negative Control}}

\author{
    Abhinav Vaddi\\
    \texttt{avaddi2@wisc.edu}\\
    University of Wisconsin--Madison
}

\date{\today}

\begin{document}

\maketitle

\begin{abstract}
Can ecological collapse theory provide early warning indicators for financial crises?
We model the US economy as a biological ecosystem---sectors as species, supply chains
as mutualistic dependencies, and bankruptcy cascades as extinction events---and apply
three bodies of ecological theory to BEA Input-Output and FRED data: May's (1972)
stability theorem, Scheffer et al.'s (2009) critical slowing down, and network
robustness metrics. We construct an Economic Biodiversity Index (EBI), track network
spectral gap, and compute critical slowing down signals across two episodes: the 2008
Global Financial Crisis (endogenous fold bifurcation, signal expected) and the 2020
COVID shock (exogenous perturbation, no signal expected by theory). In a retrospective
analysis, two of five ecological signals behaved as theorized before 2008: spectral gap
declined monotonically 2002--2007 (Kendall $\tau = -0.87$) and housing starts variance
peaked approximately 18 months before crisis onset. Logistic regression trained on these
features achieves leave-one-out AUC = 1.000 ($p = 0.034$, permutation test) with zero
false positives in cross-validation. The 2020 negative control holds: models trained on
2008 assign near-zero crisis probability to 2015--2019, and four of five ecological
signals show no pre-crisis buildup before the exogenous COVID shock, consistent with
the theoretical prediction. Spectral gap produces zero false alarms across ten calm-period
years (2010--2019). This analysis is explicitly retrospective and limited to one positive
crisis episode; generalizability requires multi-crisis validation. We propose ecological
resilience indicators as a theoretically grounded and empirically promising direction for
systemic risk research.
\end{abstract}

\section{Introduction}

Financial crises remain notoriously difficult to predict. Standard early warning systems
rely on macroeconomic aggregates---credit growth, asset prices, current account
deficits---yet the 2008 Global Financial Crisis exposed the limitations of these
approaches: most official forecasts in 2007 projected continued growth
\citep{reinhart2009}.

Ecology faces a structurally analogous problem: detecting when a stable ecosystem is
approaching collapse. Over five decades, ecologists have developed a robust theoretical
framework for understanding stability and collapse in complex systems. \citet{may1972}
proved that complexity itself can be destabilizing---above a critical threshold of
diversity and connectance, large random systems become inherently unstable.
\citet{scheffer2009} demonstrated that many complex systems exhibit ``critical slowing
down'' before tipping points: rising variance and autocorrelation in key state variables
as the system loses resilience. \citet{acemoglu2015} showed that financial networks
exhibit phase transitions where diversification reduces individual risk up to a point,
beyond which it amplifies systemic fragility through contagion channels.
\citet{haldane2011} applied this ecological analogy directly to banking systems,
demonstrating that the same complexity-fragility tradeoff governs interbank contagion
networks.

This paper operationalizes the ecology-economy analogy by constructing ecological
metrics from economic data and testing whether they were elevated before the 2008
crisis. Critically, we also test a theoretically motivated \textit{negative control}:
the 2020 COVID shock was an exogenous perturbation rather than an endogenous approach
to a bifurcation, so ecological theory predicts that critical slowing down signals
should \textit{not} appear before 2020. This contrast---signal before an endogenous
crisis, no signal before an exogenous shock---provides stronger evidence than a single
positive episode alone.

We make no claim that these signals would have been detectable in real time or that
they generalize beyond this single positive episode. That question requires multi-crisis
validation outside the scope of this retrospective study.

\section{Methods}

\subsection{Variable Mapping}

Table~\ref{tab:mapping} presents the ecological-to-economic translation underlying our
metrics. We note one important theoretical correction relative to standard ecological
analogies: supplier-buyer relationships in IO networks are structurally closer to
\textit{mutualistic} interactions (+/+) than predator-prey interactions (+/$-$), since
both parties benefit from the transaction. This distinction matters because
\citet{allesina2012} showed that mutualistic networks destabilize more rapidly with
increasing diversity and connectance than predator-prey networks---placing economic IO
networks in a potentially more fragile stability regime than the predator-prey framing
would imply.

\begin{table}[H]
\centering
\caption{Ecological-to-economic variable mapping}
\label{tab:mapping}
\begin{tabular}{ll}
\toprule
\textbf{Ecological Concept} & \textbf{Economic Analog} \\
\midrule
Species & Sector (68 BEA industries) \\
Mutualistic dependency & Supplier-buyer relationship \\
Biomass flow & Revenue flow between sectors \\
Keystone species & Systemically important sector \\
Extinction cascade & Bankruptcy contagion \\
Ecosystem biodiversity & Sector centrality distribution (EBI) \\
Critical slowing down & Rising variance + autocorrelation \\
Spectral gap collapse & Network fragmentation \\
\bottomrule
\end{tabular}
\end{table}

\subsection{Data Sources}

\textbf{BEA Input-Output Tables.} Annual sector-level flows for 68 industries from the
Bureau of Economic Analysis (TableID 259), covering two episodes: 2002--2009 (2008
GFC) and 2015--2021 (2020 COVID negative control). Edge weights are normalized to
technical coefficients (column-normalized input shares) rather than raw dollar flows,
removing nominal GDP and inflation effects from network metrics. The 2001 dot-com
episode was excluded due to incompatible industry classification systems between
pre-2002 SIC-based and post-2002 NAICS-based BEA IO tables.

\textbf{FRED Financial Time Series.} Monthly data from the Federal Reserve Economic
Database, 1998--2022: housing starts (HOUST), bank credit (TOTBKCR), TED spread
(TEDRATE, substituting for BAMLH0A0HYM2 which returned empty via API), consumer
credit (TOTALSL), and the Federal Funds rate (FEDFUNDS). All rolling windows are
strictly trailing (no centering); detrending uses first-differencing.

\textbf{Crisis Dating.} NBER recession dates for both episodes. 2008 episode: December
2007 onset (NBER), June 2009 trough. 2020 episode: March 2020 onset, December 2020
trough. Episode types pre-registered before analysis: 2008 as endogenous fold
bifurcation (signal expected), 2020 as exogenous shock (no signal expected).

\subsection{Economic Biodiversity Index (EBI)}

For each year, we construct a directed, weighted graph $G = (V, E, W)$ where nodes
are the 68 BEA sectors and edge weights are technical coefficients. We compute
betweenness centrality for each node and form a probability distribution:

\begin{equation}
p_i = \frac{c_i}{\sum_{j=1}^{n} c_j}, \quad i = 1,\ldots,n
\end{equation}

where $c_i$ is the betweenness centrality of sector $i$. The Shannon entropy of this
distribution is:

\begin{equation}
H = -\sum_{i=1}^{n} p_i \ln p_i
\end{equation}

The Economic Biodiversity Index is:

\begin{equation}
\text{EBI} = H \times (1 - \text{top3\_share})
\end{equation}

where top3\_share is the fraction of total centrality held by the three most central
sectors. This penalizes apparent diversity by keystone concentration. The choice of
betweenness centrality and the top-3 threshold are exploratory; robustness to
eigenvector centrality and DebtRank is reserved for future work.

\subsection{Spectral Gap}

The spectral gap is defined as:

\begin{equation}
\Delta \lambda = \lambda_1 - \lambda_2
\end{equation}

where $\lambda_1, \lambda_2$ are the largest and second-largest eigenvalues (by
magnitude) of the technical-coefficient adjacency matrix. A collapsing spectral gap
indicates network fragmentation---connectivity is becoming dominated by a single mode,
reducing redundant pathways.

We note explicitly that this is a \textit{network connectivity property} and is
distinct from May's (1972) stability criterion, which concerns the leading eigenvalue
of the community (Jacobian) matrix of population dynamics near equilibrium. Computing
May's criterion properly from the IO technical-coefficient matrix requires specifying
sector adjustment dynamics and self-regulation terms; this is reserved for future work.

\subsection{Critical Slowing Down}

Following \citet{scheffer2009}, we compute two metrics on monthly FRED series using
24-month strictly trailing rolling windows:

\begin{align}
\text{Variance}_t &= \text{Var}(x_{t-23:t}) \\
\text{AR(1)}_t &= \text{Corr}(x_{t-23:t-1}, x_{t-22:t})
\end{align}

Rising variance and rising autocorrelation jointly indicate critical slowing down.
This is theoretically grounded for fold bifurcations \citep{scheffer2009}: as a system
approaches a tipping point, the dominant eigenvalue of the linearized recovery dynamics
approaches zero, causing perturbations to decay increasingly slowly. We note that CSD
is only theoretically expected for \textit{endogenous} approaches to bifurcation, not
for exogenous shocks---motivating the 2020 negative control.

\subsection{Trend Significance}

We compute Kendall's $\tau$ rank correlation between year and each signal over the
pre-crisis window for both episodes. Kendall's $\tau$ is appropriate for small samples
with potential non-normality and is the standard in the EWS literature
\citep{scheffer2009}. A positive $\tau$ indicates a rising trend; the sign is adjusted
so that positive always means ``moving toward risk'' (rising for variance signals,
declining for biodiversity/connectivity signals).

\subsection{Machine Learning Evaluation}

\textbf{Labels.} For the 2008 episode, crisis labels are assigned to 2008--2009 only.
The year 2007 is held out as an ``advance warning test year''---it receives no label
and is tested separately to assess whether models fire before the official crisis onset.

\textbf{Scaler.} A RobustScaler is fitted on pre-crisis calm years only (2002--2006)
and applied to all subsequent data. This prevents look-ahead leakage through feature
normalization.

\textbf{Evaluation.} Leave-one-out cross-validation on the 7-sample annual training
set (2002--2006 calm + 2008--2009 crisis). We report mean LOO AUC as the headline
metric, not best-seed or full-sample AUC. Statistical significance is assessed via
permutation test (1000 permutations of crisis labels).

\textbf{2020 negative control.} Models trained on the 2008 episode are applied to the
2020 episode without retraining. Theory predicts near-zero crisis probability for
2015--2019 (pre-COVID calm).

\section{Results}

\subsection{Signal Analysis: Which Ecological Metrics Work?}

Table~\ref{tab:signals} summarizes the pre-crisis behavior of all five ecological
signals across both episodes. Figure~\ref{fig:signals} shows the signal trajectories.

\begin{table}[H]
\centering
\caption{Ecological signal behavior across episodes. Kendall $\tau$ computed over
pre-crisis years. For the 2020 negative control, theory predicts no significant trend
(i.e., $|\tau|$ should be small and $p$ large).}
\label{tab:signals}
\begin{tabular}{lcccc}
\toprule
& \multicolumn{2}{c}{\textbf{2008 GFC (expect signal)}}
& \multicolumn{2}{c}{\textbf{2020 COVID (expect no signal)}} \\
\cmidrule(lr){2-3} \cmidrule(lr){4-5}
\textbf{Signal} & $\tau$ & Verdict & $\tau$ & Verdict \\
\midrule
Spectral gap & $-0.87$ & \textcolor{green!60!black}{$\checkmark$ Declining} & $+0.20$ & \textcolor{green!60!black}{$\checkmark$ No trend} \\
Housing starts var & $+1.00$ & \textcolor{green!60!black}{$\checkmark$ Rising (peaks 2006)} & $-0.80$ & \textcolor{green!60!black}{$\checkmark$ Declining} \\
EBI & $+0.20$ & \textcolor{red!70!black}{$\times$ Wrong direction} & $+0.80$ & \textcolor{orange!80!black}{$\sim$ Rising} \\
Housing starts AR(1) & $-0.60$ & \textcolor{red!70!black}{$\times$ Wrong direction} & $-0.40$ & \textcolor{green!60!black}{$\checkmark$ No buildup} \\
TED spread var & $+0.40$ & \textcolor{orange!80!black}{$\sim$ Coincident only} & $+1.00$ & \textcolor{red!70!black}{$\times$ Slow rise} \\
\midrule
\textbf{Summary} & \multicolumn{2}{c}{2/5 signals as theorized} & \multicolumn{2}{c}{3/5 clean, 1 ambiguous, 1 FP} \\
\bottomrule
\end{tabular}
\end{table}

\begin{figure}[H]
\centering
\includegraphics[width=\textwidth]{fig1_signal_table.png}
\caption{Ecological early warning signals, 2008 GFC episode. Red shading indicates
crisis period (Dec 2007--Jun 2009). Verdicts indicate whether each signal moved in
the theoretically predicted direction pre-crisis. Two of five signals behaved as
theorized.}
\label{fig:signals}
\end{figure}

\textbf{Spectral gap} declined monotonically from 0.313 in 2002 to 0.221 in 2007
($\tau = -0.87$), indicating progressive network fragmentation in the pre-crisis
period. This is the cleanest ecological network signal.

\textbf{Housing starts variance} surged from 6,479 in 2002 to 25,323 in 2006 before
declining---approximately 18 months before official crisis onset. This signal is
methodologically clean: native monthly resolution, no interpolation.

\textbf{EBI} moved in the wrong direction. \textbf{Housing starts AR(1)} also went
the wrong direction (becoming more negative, not rising toward 1). \textbf{TED spread
variance} was elevated only at crisis onset---a coincident indicator, not a leading one.

\subsection{The 2020 Negative Control}

\begin{figure}[H]
\centering
\includegraphics[width=\textwidth]{fig2_endogenous_vs_exogenous.png}
\caption{Signal comparison: 2008 GFC (endogenous fold, signal expected) versus 2020
COVID (exogenous shock, no signal expected). The contrast is consistent with theoretical
predictions for both working signals.}
\label{fig:negcontrol}
\end{figure}

For spectral gap, the 2020 episode shows no directional trend pre-crisis
($\tau = +0.20$), compared to the monotonic decline before 2008 ($\tau = -0.87$).
For housing starts variance, pre-COVID values were declining 2015--2019---the opposite
of the pre-2008 buildup. These results are consistent with the theoretical prediction
that CSD signals should be specific to endogenous bifurcation approaches.

\subsection{False Positive Analysis}

We test whether the spectral gap signal fires during calm periods (2010--2014 and
2015--2019). Spectral gap produces \textbf{zero false alarms} across ten calm-period
years.

\begin{figure}[H]
\centering
\includegraphics[width=\textwidth]{fig5_false_positive_analysis.png}
\caption{False positive analysis. Spectral gap across all periods. Zero false alarms
in ten calm-period years.}
\label{fig:fp}
\end{figure}

\subsection{Trend Significance}

\begin{figure}[H]
\centering
\includegraphics[width=\textwidth]{fig6_kendall_tau.png}
\caption{Kendall $\tau$ trend statistics across both episodes. Results are consistent
with theoretical predictions for spectral gap and housing starts variance.}
\label{fig:kendall}
\end{figure}

\subsection{Keystone Sector Identification}

Betweenness centrality analysis of the 2006 network using technical coefficient
normalization identifies: Federal government enterprises (0.077), Chemical products
(0.072), and \textbf{Federal Reserve banks, credit intermediation, and related
activities (0.067)}. Finance correctly appears as the third most central sector,
correcting the raw-flow specification failure where nominal volume effects caused
Utilities and Chemicals to dominate.

\begin{figure}[H]
\centering
\includegraphics[width=\textwidth]{fig4_network_structure.png}
\caption{Network structure analysis. Left: keystone sectors, 2006. Right: spectral
gap across episodes.}
\label{fig:network}
\end{figure}

\subsection{Predictive Performance}

\begin{table}[H]
\centering
\caption{Model performance. LOO cross-validation. RobustScaler fitted on pre-crisis
years only. 2007 excluded from training labels.}
\label{tab:models}
\begin{tabular}{lccc}
\toprule
\textbf{Model} & \textbf{AUC} & \textbf{Evaluation} & \textbf{n} \\
\midrule
\textbf{Logistic Regression (primary)} & \textbf{1.000} & LOO CV & 7 \\
Random Forest & 0.400 & LOO CV & 7 \\
\midrule
\multicolumn{4}{l}{\textit{Permutation test (LR): $p = 0.034$, null mean = 0.483}} \\
\multicolumn{4}{l}{\textit{Advance warning test (2007): LR = 0.013, RF = 0.120}} \\
\multicolumn{4}{l}{\textit{2020 negative control: LR assigns 0.002--0.004 pre-crisis}} \\
\midrule
GNN (appendix only) & $0.361 \pm 0.440$ & 5 seeds & 7 \\
\bottomrule
\end{tabular}
\vspace{4pt}

\textit{Note: LOO AUC = 1.000 on n=7 has wide confidence intervals. The permutation
test ($p = 0.034$) provides limited but non-trivial evidence. Multi-crisis validation
is required. Random Forest underperforms, consistent with overfitting at small n.}
\end{table}

\begin{figure}[H]
\centering
\includegraphics[width=\textwidth]{fig7_results_summary.png}
\caption{Honest claims and evidence summary.}
\label{fig:results}
\end{figure}

\textbf{Advance warning test.} Neither model fires on 2007 (LR: 0.013, RF: 0.120).
This is an honest null result.

\textbf{2020 negative control (ML).} Logistic regression assigns 0.002--0.004 crisis
probability to 2015--2019 with zero false alarms---consistent with the theoretical
prediction that ecological signals are quiet before an exogenous shock.

\section{Discussion}

\subsection{Summary of Findings}

\begin{enumerate}
\item Two of five ecological signals---spectral gap ($\tau = -0.87$) and housing starts
variance (peaks 18 months pre-crisis)---behaved as theorized before the 2008 GFC.

\item The 2020 COVID negative control holds for four of five signals and for the ML
model: no pre-crisis ecological buildup before an exogenous shock.

\item Spectral gap produces zero false alarms across ten calm-period years.

\item Logistic regression achieves LOO AUC = 1.000 ($p = 0.034$); neither model
provides advance warning in the year before crisis onset.

\item Three signals did not behave as theorized, providing honest evidence about
the limits of the ecological mapping.
\end{enumerate}

\subsection{What the Results Do and Do Not Show}

These results support the ecological framework as theoretically motivated and
empirically promising for two specific signals. They do not demonstrate actionable
real-time early warning, generalizability across crises, or incremental value beyond
standard macroeconomic indicators.

The most meaningful result is the combination: spectral gap works before an endogenous
crisis and is quiet before an exogenous one---precisely the discriminating pattern
the theory predicts.

\subsection{The Wrong Network Problem}

The BEA IO network captures supply-chain flows, not financial exposure networks. The
2008 crisis propagated through mortgage securitization, interbank lending, and CDS
exposures---none of which appear in IO tables. Applying ecological metrics to a
financial exposure network from FFIEC Call Reports is the most important methodological
improvement for future work.

\subsection{The Theoretical Agenda}

This paper computes the spectral gap of the adjacency matrix rather than the leading
eigenvalue of the economic community (Jacobian) matrix---May's actual criterion.
Properly computing May's criterion requires a dynamic Leontief-type model and a
Jacobian with self-regulation terms on the diagonal. The leading eigenvalue of this
Jacobian rising toward zero would unify the network fragmentation and CSD signals:
both are manifestations of the same underlying quantity. This unification is the most
important theoretical contribution available to this research program.

\subsection{Limitations}
\label{sec:limitations}

\textbf{Single positive episode.} All metrics have wide confidence intervals.
Multi-crisis leave-one-out validation is required.

\textbf{Interpolation leakage.} Cubic spline interpolation of annual BEA data uses
future anchor points. Primary results use annual-resolution network metrics and native
monthly CSD signals only.

\textbf{Wrong network.} IO network captures supply-chain flows, not financial
contagion channels.

\textbf{Signals that failed.} EBI, housing AR(1), and TED spread variance did not
behave as theorized. These failures constrain which ecological mechanisms are supported.

\textbf{Robustness.} Results are robust to crisis onset dating (AUC = 1.000 under
both December 2007 and January 2008 onset specifications). Housing starts variance
Kendall $\tau$ is positive across all window lengths tested (12--36 months), ranging
from 0.60 to 1.00, and is statistically significant at $p < 0.05$ for windows of
24 months or longer. CSD signals alone achieve LOO AUC = 1.000; network-only
achieves 0.400.

\subsection{Future Work}

\begin{enumerate}
\item \textbf{Multi-crisis panel.} Add 2001 episode and cross-country panels;
evaluate with leave-one-crisis-out.

\item \textbf{Financial exposure network.} Construct from FFIEC Call Reports;
apply EBI, spectral gap, and DebtRank.

\item \textbf{Economic community matrix.} Compute May's actual stability criterion;
unify network and CSD signals under a single eigenvalue framework.

\item \textbf{Real-time data.} Pull ALFRED real-time vintages.

\item \textbf{Benchmark comparison.} Test incremental value beyond credit-to-GDP
gap \citep{borio2002} and yield-curve inversion.

\item \textbf{GNN on actual sector graph.} 68-node IO graph with spatiotemporal
architecture; pretrain on real IUCN extinction cascade data.
\end{enumerate}

\subsection{Conclusion}

The ecological framework for systemic risk monitoring is theoretically motivated and
produces two signals that behave as theorized. The 2020 negative control provides
discriminating evidence: ecological signals are quiet before an exogenous shock and
elevated before an endogenous one, consistent with fold bifurcation theory. Whether
these signals provide actionable, generalizable early warning value beyond standard
indicators remains open. We propose them as a theoretically grounded direction, with
the methodological agenda above as the necessary next steps.

\section*{Acknowledgments}

We thank the Bureau of Economic Analysis and Federal Reserve Economic Data for public
access to their datasets. All code is available at
\url{https://github.com/Abhiv1028/EcoEcon}.

\appendix
\section{GNN Architecture and Full Results}
\label{app:gnn}

The GNN is a preliminary methodology demonstration, not a primary result.

We construct a graph with 8 nodes (features) and edges connecting features with
Pearson correlation $|r| > 0.3$, computed on pre-crisis training data only.

\begin{table}[H]
\centering
\caption{GNN results (appendix). Not primary findings.}
\begin{tabular}{lcc}
\toprule
\textbf{Model} & \textbf{AUC} & \textbf{Evaluation} \\
\midrule
GNN (no pretraining) & reported in notebook & LOO, 5 seeds \\
GNN (synthetic ecological pretraining) & reported in notebook & LOO, 5 seeds \\
\midrule
\multicolumn{3}{l}{\textit{Best-seed AUC = 0.993 selected by validation performance.}} \\
\multicolumn{3}{l}{\textit{Not an unbiased estimate. See notebook 03 for full results.}} \\
\bottomrule
\end{tabular}
\end{table}

The GNN uses the 8-node feature-correlation graph rather than the actual 68-node IO
sector graph. Future work will use the actual sector graph with a spatiotemporal
architecture and real IUCN extinction cascade pretraining.

\begin{thebibliography}{99}

\bibitem{may1972}
May, R.M. (1972).
Will a large complex system be stable?
\textit{Nature}, 238, 413--414.

\bibitem{scheffer2009}
Scheffer, M., Bascompte, J., Brock, W.A., et al. (2009).
Early-warning signals for critical transitions.
\textit{Nature}, 461, 53--59.

\bibitem{acemoglu2015}
Acemoglu, D., Ozdaglar, A., \& Tahbaz-Salehi, A. (2015).
Systemic risk and stability in financial networks.
\textit{American Economic Review}, 105(2), 564--608.

\bibitem{haldane2011}
Haldane, A.G. \& May, R.M. (2011).
Systemic risk in banking ecosystems.
\textit{Nature}, 469, 351--355.

\bibitem{allesina2012}
Allesina, S. \& Tang, S. (2012).
Stability criteria for complex ecosystems.
\textit{Nature}, 483, 205--208.

\bibitem{shannon1948}
Shannon, C.E. (1948).
A mathematical theory of communication.
\textit{Bell System Technical Journal}, 27, 379--423.

\bibitem{hamilton2017}
Hamilton, W.L., Ying, R., \& Leskovec, J. (2017).
Inductive representation learning on large graphs.
\textit{Advances in Neural Information Processing Systems}, 30.

\bibitem{reinhart2009}
Reinhart, C.M. \& Rogoff, K.S. (2009).
\textit{This Time Is Different: Eight Centuries of Financial Folly}.
Princeton University Press.

\bibitem{borio2002}
Borio, C. \& Lowe, P. (2002).
Asset prices, financial and monetary stability: exploring the nexus.
\textit{BIS Working Papers}, No. 114.

\bibitem{diamond1983}
Diamond, D.W. \& Dybvig, P.H. (1983).
Bank runs, deposit insurance, and liquidity.
\textit{Journal of Political Economy}, 91(3), 401--419.

\bibitem{dakos2012}
Dakos, V., Carpenter, S.R., Brock, W.A., et al. (2012).
Methods for detecting early warnings of critical transitions in time series.
\textit{PLOS ONE}, 7(7), e41010.

\end{thebibliography}

\end{document}
'''

# ── Write .tex file ───────────────────────────────────────────
tex_path = os.path.expanduser('~/ecoecon/paper/ecoecon_paper.tex')
os.makedirs(os.path.dirname(tex_path), exist_ok=True)
with open(tex_path, 'w') as f:
    f.write(latex_content)
print(f"LaTeX written to: {tex_path}")

# ── Recompile PDF ─────────────────────────────────────────────
print("\n=== COMPILING PDF ===")
paper_dir = os.path.expanduser('~/ecoecon/paper')
for run in range(2):
    result = subprocess.run(
        ['pdflatex', '-interaction=nonstopmode', 'ecoecon_paper.tex'],
        cwd=paper_dir, capture_output=True, text=True
    )
    status = '✓' if result.returncode == 0 else '✗'
    print(f"  Run {run+1}: {status}")
    if result.returncode != 0:
        errors = [l for l in result.stdout.split('\n')
                  if 'error' in l.lower() or 'Error' in l]
        for e in errors[:5]:
            print(f"    {e}")

pdf_path = os.path.join(paper_dir, 'ecoecon_paper.pdf')
if os.path.exists(pdf_path):
    size = os.path.getsize(pdf_path) / 1024
    print(f"  PDF: {size:.0f} KB")

# ── Commit ────────────────────────────────────────────────────
print("\n=== COMMITTING ===")
root = os.path.expanduser('~/ecoecon')
commands = [
    ['git', '-C', root, 'add', '.'],
    ['git', '-C', root, 'commit', '-m',
     'Final: clean LaTeX, recompiled PDF, natbib fix'],
    ['git', '-C', root, 'push'],
]
for cmd in commands:
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(' '.join(cmd[-2:]))
    print(r.stdout or r.stderr)
    print('---')

LaTeX written to: /Users/abhinavvaddi/ecoecon/paper/ecoecon_paper.tex

=== COMPILING PDF ===
  Run 1: ✗
    ! Package pdftex.def Error: File `fig1_signal_table.png' not found: using draft
    ! Package pdftex.def Error: File `fig2_endogenous_vs_exogenous.png' not found: 
    ! Package pdftex.def Error: File `fig5_false_positive_analysis.png' not found: 
    ! Package pdftex.def Error: File `fig6_kendall_tau.png' not found: using draft 
    ! Package pdftex.def Error: File `fig4_network_structure.png' not found: using 
  Run 2: ✗
    ! Package pdftex.def Error: File `fig1_signal_table.png' not found: using draft
    ! Package pdftex.def Error: File `fig2_endogenous_vs_exogenous.png' not found: 
    ! Package pdftex.def Error: File `fig5_false_positive_analysis.png' not found: 
    ! Package pdftex.def Error: File `fig6_kendall_tau.png' not found: using draft 
    ! Package pdftex.def Error: File `fig4_network_structure.png' not found: using 
  PDF: 203 KB

=== COMMITTING ===
add .

---
